[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/editorial-v2/notebooks/14_Breathing_Dynamics.ipynb)

# Notebook 14 — Breathing Dynamics

**Companion to Chapter 14**

This laboratory treats breathing as periodic buoyancy forcing and as a bounded, delayed actuator. It is a systems model—not breathing or diving advice.

## Learning objectives

- convert lung-volume variation into buoyant force;
- simulate a first-order lung-volume actuator;
- couple tidal and deliberate volume changes to vertical motion;
- explore slow BCD trim and fast breathing correction.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})
rho, g, p0 = 1025.0, 9.80665, 101325.0

def ambient_pressure_pa(depth_m):
    return p0 + rho*g*np.asarray(depth_m)

def ambient_pressure_bar(depth_m):
    return ambient_pressure_pa(depth_m)/1e5

## 1. From tidal volume to buoyant force

The incremental force is $\Delta F_B=\rho g\Delta V_L$.

In [ ]:
tidal_l = np.array([0.25,0.35,0.50,0.75])
force_n = rho*g*tidal_l*1e-3
assert np.allclose(force_n, rho*g*tidal_l*1e-3)
for volume, force in zip(tidal_l,force_n):
    print(f"{volume:.2f} L -> {force:.2f} N")

## 2. First-order bounded actuator

The deliberate component obeys $\tau_L\dot V_C+V_C=\operatorname{sat}(V_{cmd})$.

In [ ]:
def simulate_lung(command_l=0.4,tau=0.7,limit_l=0.5,duration=6):
    t=np.linspace(0,duration,601)
    target=np.clip(command_l,-limit_l,limit_l)*1e-3
    sol=solve_ivp(lambda t,x:[(target-x[0])/tau],(0,duration),[0],t_eval=t)
    return t,sol.y[0]

t,vc=simulate_lung()
assert np.max(np.abs(vc)) <= 0.5e-3 + 1e-12
assert np.isclose(vc[-1], 0.4e-3, rtol=2e-3)
plt.plot(t,1e3*vc)
plt.xlabel("Time [s]"); plt.ylabel("Controlled volume [L]")
plt.title("Bounded lung-volume response"); plt.show()

## 3. Coupled vertical model

Depth is positive downward and velocity is positive upward. The model includes the unstable local buoyancy slope from Part II, linear drag, tidal breathing, and controlled lung volume.

In [ ]:
m=85.0
z_star=20.0
vg0=8e-3
p_star=ambient_pressure_pa(z_star)
vg_star=vg0*p0/p_star
k_b=-rho**2*g**2*vg_star/p_star
c=45.0

def simulate_vertical(feedback=False,bcd_trim=False,duration=45,dt=0.01):
    t=np.arange(0,duration+dt,dt); x=np.zeros((4,len(t)))
    x[:,0]=[0.40,0,0,0]  # depth error, upward velocity, lung control, BCD volume
    tidal=0.35e-3*np.sin(2*np.pi*0.25*t)
    for k in range(len(t)-1):
        z,v,vl,vb=x[:,k]
        lung_cmd=np.clip((0.00055*z-0.0015*v) if feedback else 0,-0.5e-3,0.5e-3)
        bcd_cmd=np.clip(0.00035*z,-1.5e-3,1.5e-3) if bcd_trim else 0
        dz=-v
        dv=(k_b*z-c*v+rho*g*(tidal[k]+vl+vb))/m
        dvl=(lung_cmd-vl)/0.7
        dvb=(bcd_cmd-vb)/8.0
        x[:,k+1]=x[:,k]+dt*np.array([dz,dv,dvl,dvb])
    return t,x,tidal

t,x0,tidal=simulate_vertical()
t,x1,_=simulate_vertical(feedback=True)
t,x2,_=simulate_vertical(feedback=True,bcd_trim=True)
assert np.isfinite(x0).all() and np.isfinite(x1).all() and np.isfinite(x2).all()
plt.plot(t,x0[0],label="tidal only")
plt.plot(t,x1[0],label="breathing feedback")
plt.plot(t,x2[0],label="breathing + slow trim")
plt.xlabel("Time [s]"); plt.ylabel("Depth error [m]")
plt.title("Actuator allocation changes vertical motion"); plt.legend(); plt.show()

## 4. Frequency response of the lung actuator

In [ ]:
tau=0.7
w=np.logspace(-2,2,500)
mag=1/np.sqrt(1+(w*tau)**2)
phase=-np.arctan(w*tau)*180/np.pi
fig,ax=plt.subplots(2,1,sharex=True,figsize=(8,6))
ax[0].semilogx(w,20*np.log10(mag)); ax[0].set_ylabel("Magnitude [dB]")
ax[1].semilogx(w,phase); ax[1].set(xlabel="Angular frequency [rad/s]",ylabel="Phase [deg]")
plt.show()

## Engineering exercises

1. Sweep tidal amplitude and measure steady depth oscillation.
2. Change $\tau_L$ and compare phase lag at $0.25$ Hz.
3. Find a feedback gain that causes oscillation and explain the phase mechanism.
4. Design a smoother transfer of sustained correction to BCD trim.


In [ ]:
# Exercise starter: sweep tidal volume and measure depth oscillation
exercise_tidal_l = np.array([0.20, 0.35, 0.50])
# Adapt simulate_vertical() to accept tidal amplitude and compare responses.


## Summary

Breathing contributes a periodic disturbance and a limited control channel. Adding its actuator state exposes lag, saturation, and the need to allocate slow and fast correction coherently.